# Spectral Mixture Kernel — 1D Interactive Visualisation

Plots $k(0, \Delta)$ as a function of lag $\Delta$ for $Q$ mixture components.

The kernel follows the `[0,1]`-domain convention used in `simulation.py`:
$$k(\Delta) = \sum_{q=1}^{Q} w_q \exp\!\left(-2\pi^2 v_q \Delta^2\right) \cos\!\left(2\pi \mu_q \Delta\right)$$
where $v_q = 1/(2\pi^2 \ell_q^2)$, so $\ell_q$ is the lengthscale of component $q$ in $[0,1]$ units.

Dashed coloured lines show individual components; the solid black line is their normalised sum.

**Dependencies:** `numpy`, `matplotlib`, `ipywidgets` — all standard in JupyterLab, no extra installs needed.

In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

In [2]:
# ---------------------------------------------------------------------------
# SM_CONFIGS — same values as simulation.py, [0,1]-domain parameterisation
# ---------------------------------------------------------------------------

SM_CONFIGS = {
    'slow': {
        'weights':   [0.5, 0.3, 0.2],
        'means':     [[0.5, 0.4], [1.0, 0.8], [1.5, 1.2]],
        'variances': [[2.0, 2.0], [3.0, 2.5], [4.0, 3.0]],
    },
    'medium': {
        'weights':   [0.5, 0.3, 0.2],
        'means':     [[1.0, 0.8], [2.5, 1.5], [4.0, 3.0]],
        'variances': [[4.0, 3.0], [6.0, 5.0], [8.0, 6.0]],
    },
    'fast': {
        'weights':   [0.5, 0.3, 0.2],
        'means':     [[2.0, 1.5], [5.0, 3.5], [8.0, 6.0]],
        'variances': [[6.0, 5.0], [8.0, 6.0], [12.0, 10.0]],
    },
}

COMPONENT_COLOURS = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0']


# ---------------------------------------------------------------------------
# Kernel maths
# ---------------------------------------------------------------------------

def ell_to_var(ell):
    return 1.0 / (2.0 * np.pi**2 * ell**2 + 1e-12)

def sm_component(delta, w, mu, ell):
    v   = ell_to_var(ell)
    env = np.exp(-2.0 * np.pi**2 * v * delta**2)
    osc = np.cos(2.0 * np.pi * mu * delta)
    return w * env * osc

def sm_kernel(delta, weights, means, ells):
    return sum(sm_component(delta, w, mu, ell)
               for w, mu, ell in zip(weights, means, ells))

In [4]:
# ---------------------------------------------------------------------------
# Configuration — edit these to change the initial state
# ---------------------------------------------------------------------------

Q         = 3      # number of mixture components (1–4)
DELTA_MAX = 1.0    # maximum lag shown on x-axis
CONFIG    = None   # None for generic defaults, or 'slow'/'medium'/'fast'


def initial_params(Q, config_name=None):
    if config_name is not None:
        cfg = SM_CONFIGS[config_name]
        q   = min(Q, len(cfg['weights']))
        ws  = list(cfg['weights'][:q])
        mus = [cfg['means'][i][0] for i in range(q)]
        els = [float(1.0 / np.sqrt(2.0 * np.pi**2 * cfg['variances'][i][0] + 1e-12))
               for i in range(q)]
        while len(ws) < Q:
            ws.append(0.2); mus.append(1.0); els.append(0.2)
        return ws[:Q], mus[:Q], els[:Q]
    else:
        return ([round(1.0 / Q, 3)] * Q,
                [round(0.5 + i * 1.5, 2) for i in range(Q)],
                [0.2] * Q)


init_w, init_mu, init_ell = initial_params(Q, CONFIG)
delta = np.linspace(0.0, DELTA_MAX, 1000)

In [5]:
# ---------------------------------------------------------------------------
# Interactive widget
# ---------------------------------------------------------------------------

slider_style  = {'description_width': '60px'}
slider_layout = widgets.Layout(width='400px')

w_sliders   = [widgets.FloatSlider(value=init_w[q],   min=0.01, max=2.0,  step=0.01,
                                   description='weight', readout_format='.2f',
                                   style=slider_style, layout=slider_layout)
               for q in range(Q)]
mu_sliders  = [widgets.FloatSlider(value=init_mu[q],  min=0.05, max=12.0, step=0.05,
                                   description='freq μ', readout_format='.2f',
                                   style=slider_style, layout=slider_layout)
               for q in range(Q)]
ell_sliders = [widgets.FloatSlider(value=init_ell[q], min=0.02, max=1.0,  step=0.01,
                                   description='ℓ', readout_format='.2f',
                                   style=slider_style, layout=slider_layout)
               for q in range(Q)]

panels = []
for q in range(Q):
    col   = COMPONENT_COLOURS[q % len(COMPONENT_COLOURS)]
    label = widgets.HTML(f'<b style="color:{col}">Component {q+1}</b>')
    panel = widgets.VBox(
        [label, w_sliders[q], mu_sliders[q], ell_sliders[q]],
        layout=widgets.Layout(border=f'1.5px solid {col}',
                              padding='8px 12px', margin='4px')
    )
    panels.append(panel)

controls = widgets.HBox(panels)


# Plot output widget — redraws on every slider change
plot_out = widgets.Output()

def redraw(*args):
    ws  = [sl.value for sl in w_sliders]
    mus = [sl.value for sl in mu_sliders]
    els = [sl.value for sl in ell_sliders]

    w_sum = sum(ws) or 1.0
    w_n   = [w / w_sum for w in ws]

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.axhline(0, color='0.8', linewidth=0.8, zorder=0)
    ax.set_xlim(0, DELTA_MAX)
    ax.set_xlabel('Lag  Δ = x − x′', fontsize=12)
    ax.set_ylabel('k(0, Δ)', fontsize=12)
    ax.set_title('Spectral Mixture Kernel — 1D', fontsize=13)

    all_vals = []
    for q in range(Q):
        col  = COMPONENT_COLOURS[q % len(COMPONENT_COLOURS)]
        vals = sm_component(delta, w_n[q], mus[q], els[q])
        ax.plot(delta, vals, color=col, alpha=0.45, linewidth=1.4,
                linestyle='--', label=f'Component {q+1}')
        all_vals.append(vals)

    total = sm_kernel(delta, w_n, mus, els)
    ax.plot(delta, total, color='black', linewidth=2.0, label='Sum')
    all_vals.append(total)

    combined = np.concatenate(all_vals)
    pad = 0.1 * (combined.max() - combined.min() + 1e-6)
    ax.set_ylim(combined.min() - pad, combined.max() + pad)
    ax.legend(loc='upper right', fontsize=9, framealpha=0.7)
    fig.tight_layout()

    plot_out.clear_output(wait=True)
    with plot_out:
        plt.show()
    plt.close(fig)


for sl in w_sliders + mu_sliders + ell_sliders:
    sl.observe(redraw, names='value')

redraw()  # draw initial plot
display(widgets.VBox([controls, plot_out]))